# Experiment planning: sample size and MDE

Two complementary questions:

- **`calculate_sample_size`**: How many participants do we need to detect a target effect?
- **`calculate_mde`**: What effect can we detect with a fixed number of participants?

Both functions use one named outcome and historical control data. Omit covariates for classic planning; pass an explicit list to enable CUPED.

In [1]:
from causalis.shared.rct_design import calculate_mde, calculate_sample_size
from causalis.dgp import generate_rct_causal_data
import pandas as pd

## Historical experiment

Generate reproducible revenue data with a positive baseline. `prior_spend` is a pre-treatment covariate associated with revenue. Both planners estimate the baseline and variance using historical **control rows only**.

In [2]:
data = generate_rct_causal_data(
    n=20_000,
    n_treatments=2,
    d_names=["control", "variant"],
    confounder_specs=[{"name": "prior_spend", "dist": "normal"}],
    outcome_specs=[
        {"name": "revenue", "alpha_y": 20, "beta_y": [3], "theta": [1]},
    ],
    seed=42,
)

data

RctCausalData(df=(20000, 5), treatments=['control', 'variant'], control_treatment='control', outcomes=['revenue'], confounders=['prior_spend'], user_id='user_id')

## 1. Target effect → required sample size

The default scenario compares relative MDEs **[0.5, 1, 5, 10, 20]%**. Each row below is a separate candidate experiment with an equal control/variant split.

No covariates are supplied, so this uses classic sample variance even though `data` declares `prior_spend`.

In [3]:
sizes = calculate_sample_size(data, outcome="revenue")
sizes

,outcome,control,treatment,mde_relative,mde_absolute,n_control,n_treatment,n_total
0,revenue,control,variant,0.5,0.099862,15623,15623,31246
1,revenue,control,variant,1.0,0.199724,3906,3906,7812
2,revenue,control,variant,5.0,0.998622,157,157,314
3,revenue,control,variant,10.0,1.997244,40,40,80
4,revenue,control,variant,20.0,3.994487,10,10,20


### Custom relative and absolute targets

For `scenario="user"`, provide both `mde_type` and a list of positive targets. Relative inputs are percentages: **1 means 1%**. Absolute inputs are differences in revenue units. For a baseline of 20, a 1% relative target corresponds to an absolute difference of 0.2.

In [4]:
relative_sizes = calculate_sample_size(
    data,
    outcome="revenue",
    scenario="user",
    mde_type="relative",
    mde=[1, 3, 5],
)
relative_sizes

,outcome,control,treatment,mde_relative,mde_absolute,n_control,n_treatment,n_total
0,revenue,control,variant,1.0,0.199724,3906,3906,7812
1,revenue,control,variant,3.0,0.599173,434,434,868
2,revenue,control,variant,5.0,0.998622,157,157,314


In [5]:
absolute_sizes = calculate_sample_size(
    data,
    outcome="revenue",
    scenario="user",
    mde_type="absolute",
    mde=[0.2, 0.5, 1.0],
)
absolute_sizes

,outcome,control,treatment,mde_relative,mde_absolute,n_control,n_treatment,n_total
0,revenue,control,variant,1.00138,0.2,3895,3895,7790
1,revenue,control,variant,2.50345,0.5,624,624,1248
2,revenue,control,variant,5.00690,1.0,156,156,312


### Compare classic and CUPED sample sizes

Use the same percentage targets and allocation for both methods. Only CUPED receives an explicit covariate list. Historical variance is estimated once per call and reused across targets.

In [6]:
cuped_sizes = calculate_sample_size(
    data,
    outcome="revenue",
    covariates=["prior_spend"],
)

size_comparison = pd.concat(
    [sizes.assign(method="Classic"), cuped_sizes.assign(method="CUPED")],
    ignore_index=True,
)[["method", "mde_relative", "mde_absolute", "n_control", "n_treatment", "n_total"]]
size_comparison.sort_values("mde_relative", kind="stable").reset_index(drop=True)

,method,mde_relative,mde_absolute,n_control,n_treatment,n_total
0,Classic,0.5,0.099862,15623,15623,31246
1,CUPED,0.5,0.099862,1570,1570,3140
2,Classic,1.0,0.199724,3906,3906,7812
3,CUPED,1.0,0.199724,393,393,786
4,Classic,5.0,0.998622,157,157,314
5,CUPED,5.0,0.998622,16,16,32
6,Classic,10.0,1.997244,40,40,80
7,CUPED,10.0,1.997244,4,4,8
8,Classic,20.0,3.994487,10,10,20
9,CUPED,20.0,3.994487,1,1,2


## 2. Fixed audience → detectable effect

`sample_size` is the **total** number of future participants across all groups. The following calls use the same 10,000-person audience and equal allocation. Both return absolute and relative MDE; only the second call enables CUPED.

In [7]:
classic_mde = calculate_mde(
    data,
    outcome="revenue",
    sample_size=10_000,
)

cuped_mde = calculate_mde(
    data,
    outcome="revenue",
    sample_size=10_000,
    covariates=["prior_spend"],
)

pd.concat(
    [classic_mde.assign(method="Classic"), cuped_mde.assign(method="CUPED")],
    ignore_index=True,
)[["method", "mde_absolute", "mde_relative", "n_control", "n_treatment", "n_total"]]

,method,mde_absolute,mde_relative,n_control,n_treatment,n_total
0,Classic,0.176521,0.883824,5000,5000,10000
1,CUPED,0.055944,0.280108,5000,5000,10000


## 3. Allocation and diagnostic details

Allocation keys match declared treatment groups and shares must sum to one. MDE calculation preserves the exact total using largest remainders; ties follow contract order, control first. Each group must receive at least one participant.

`include_details=True` adds the baseline, raw and used variances, variance reduction, historical control size, significance, and power. Sample-size results also include achieved power after upward rounding.

In [8]:
detailed_mde = calculate_mde(
    data,
    outcome="revenue",
    sample_size=10_001,
    covariates=["prior_spend"],
    allocation={"control": 0.6, "variant": 0.4},
    include_details=True,
)
detailed_mde

,outcome,control,treatment,mde_relative,mde_absolute,n_control,n_treatment,n_total,baseline_mean,variance_raw,variance_used,variance_reduction_pct,n_reference,alpha,power
0,revenue,control,variant,0.285875,0.057096,6001,4000,10001,19.972437,9.924925,0.996891,89.955679,9988,0.05,0.8


In [9]:
detailed_sizes = calculate_sample_size(
    data,
    outcome="revenue",
    covariates=["prior_spend"],
    scenario="user",
    mde_type="relative",
    mde=[1],
    allocation={"control": 0.6, "variant": 0.4},
    include_details=True,
)
detailed_sizes

,outcome,control,treatment,mde_relative,mde_absolute,n_control,n_treatment,n_total,baseline_mean,variance_raw,variance_used,variance_reduction_pct,n_reference,alpha,power,achieved_power
0,revenue,control,variant,1.0,0.199724,491,327,818,19.972437,9.924925,0.996891,89.955679,9988,0.05,0.8,0.800256


## 4. Format for presentation

Keep numeric results unchanged and format a separate display copy. MDE columns contain **requested targets** in sample-size results and **calculated sensitivity** in MDE results.

In [10]:
presented = sizes.copy()
presented["mde_relative"] = presented["mde_relative"].map("{:.1f}%".format)
presented["mde_absolute"] = presented["mde_absolute"].map("{:.3f}".format)
for column in ["n_control", "n_treatment", "n_total"]:
    presented[column] = presented[column].map("{:,.0f}".format)
presented

,outcome,control,treatment,mde_relative,mde_absolute,n_control,n_treatment,n_total
0,revenue,control,variant,0.5%,0.100,"15,623","15,623","31,246"
1,revenue,control,variant,1.0%,0.200,"3,906","3,906","7,812"
2,revenue,control,variant,5.0%,0.999,157,157,314
3,revenue,control,variant,10.0%,1.997,40,40,80
4,revenue,control,variant,20.0%,3.994,10,10,20


## Interpretation

- Omitted covariates, `None`, and `[]` use classic planning without regression. Explicit covariates enable CUPED.
- Relative values use the raw historical control mean. Relative target planning requires a positive mean; absolute planning and MDE calculation remain available otherwise, with relative output set to NaN.
- `n_total` includes control once. Do not sum it across active-arm comparisons within a candidate experiment.
- Planning assumes independent randomization units, two-sided normal power, and the same historical-control variance in future arms. Binary outcomes use the same fixed-variance approximation.
- Power is per comparison, without a multiple-testing correction or joint detection guarantee. Small calculated samples remain normal-approximation planning estimates.